In [27]:
#### Tratamento e exploração dos dados das bases ###
import pandas as pd
import numpy as np
import uuid
import matplotlib.pyplot as plt

def load_hotelbooking_data():
    df = pd.read_csv('D:\Projetos\Visão de Cliente Único\Hotel Booking\hotel_bookings.csv')
    df['children'] = df['children'].fillna(0).astype(int)
    df['company']  = df['company'].fillna(0).astype(int) 
    df['agent']    = df['agent'].fillna(0).astype(int)
    # Converter colunas do tipo object para string para otimização de memória
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('string')
    df = create_unique_id(df, 'hotelbooking_id') 
    return df

#### Função para carregar e mesclar dados do E-commerce ####
def load_ecommerce_data():
    folder_path = r'D:\Projetos\Visão de Cliente Único\E-commerce'
    # Carregar os datasets
    customers = pd.read_csv(f'{folder_path}/olist_customers_dataset.csv')
    orders = pd.read_csv(f'{folder_path}/olist_orders_dataset.csv')
    items = pd.read_csv(f'{folder_path}/olist_order_items_dataset.csv')
    products = pd.read_csv(f'{folder_path}/olist_products_dataset.csv')
    sellers = pd.read_csv(f'{folder_path}/olist_sellers_dataset.csv')
    reviews = pd.read_csv(f'{folder_path}/olist_order_reviews_dataset.csv')
    payments = pd.read_csv(f'{folder_path}/olist_order_payments_dataset.csv')
    
    # Selecionar colunas relevantes
    customers_sel = customers[['customer_id', 'customer_city']]
    orders_sel = orders[['order_id', 'customer_id', 'order_purchase_timestamp']]
    items_sel = items[['order_id', 'product_id', 'seller_id', 'freight_value']]
    products_sel = products[['product_id', 'product_category_name']]
    sellers_sel = sellers[['seller_id', 'seller_city']]
    reviews_sel = reviews[['order_id', 'review_score']]
    payments_sel = payments[['order_id', 'payment_type', 'payment_installments', 'payment_value']]
    
    # Mesclar os dataframes
    df = orders_sel.merge(customers_sel, on='customer_id', how='left')
    df = df.merge(items_sel, on='order_id', how='left')
    df = df.merge(products_sel, on='product_id', how='left')
    df = df.merge(sellers_sel, on='seller_id', how='left')
    df = df.merge(reviews_sel, on='order_id', how='left')
    df = df.merge(payments_sel, on='order_id', how='left')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('string')
    df = create_unique_id(df, 'ecommerce_id') 
    df = df.dropna(subset=['product_category_name','review_score','payment_type'])
    return df

def load_shopping_data():
    df = pd.read_csv('D:\Projetos\Visão de Cliente Único\Shopping\shopping_trends_updated.csv')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('string')
    df = create_unique_id(df, 'shopping_id')   
    return df

def generate_intersected_clients(df_ecommerce, df_shopping, df_hotelbooking):
    # 1. Gerando 5.000 CPFs únicos de forma aleatória
    np.random.seed(42) # Mantém os resultados consistentes
    cpfs_base = [f"{np.random.randint(100, 999)}.{np.random.randint(100, 999)}.{np.random.randint(100, 999)}-{np.random.randint(10, 99)}" for _ in range(5000)]

    # 2. Aplicando a Regra de Negócio (Segmentação de Público)
    cpfs_ecom_only = cpfs_base[:4000]       # 80%: Apenas E-commerce
    cpfs_ecom_shop = cpfs_base[4000:4750]   # 15%: E-commerce + Shopping
    cpfs_vip = cpfs_base[4750:]             # 5%: Super VIP (E-commerce + Shopping + Hotel)

    # 3. Criando as listas de CPFs que vão para cada base
    lista_cpfs_ecommerce = cpfs_ecom_only + cpfs_ecom_shop + cpfs_vip
    lista_cpfs_shopping = cpfs_ecom_shop + cpfs_vip
    lista_cpfs_hotel = cpfs_vip
    df_ecommerce['CPF'] = np.random.choice(lista_cpfs_ecommerce, size=len(df_ecommerce))
    df_shopping['CPF'] = np.random.choice(lista_cpfs_shopping, size=len(df_shopping))
    df_hotelbooking['CPF'] = np.random.choice(lista_cpfs_hotel, size=len(df_hotelbooking))

    return df_ecommerce, df_shopping, df_hotelbooking

def create_unique_id(df, nome_coluna):
    """
    Função para criar um ID único para cada reserva no DataFrame.
    Adiciona uma coluna 'reservation_id' com UUIDs únicos.
    """
    df = df.copy()  # Para evitar modificar o DataFrame original
    df[nome_coluna] = [str(uuid.uuid4()) for _ in range(len(df))]

    return df

df_hotelbooking = load_hotelbooking_data()
df_ecommerce = load_ecommerce_data()
df_shopping = load_shopping_data()

df_ecommerce, df_shopping, df_hotelbooking = generate_intersected_clients(df_ecommerce, df_shopping, df_hotelbooking)


In [ ]:
def analisar_palavra(palavra):
    if palavra == palavra[::-1]:
        return "Palavra é um palíndromo"
    return "Palavra não é um palíndromo"

palavra = "arara"
resultado = analisar_palavra(palavra)
resultado


'Palavra é um palíndromo'